# Debugging ML Systems: 6 Worked Scenarios

Debugging is a systematic skill, not intuition. This note presents the diagnostic framework MLE interviews test and walks through 6 fully worked debugging scenarios in dialogue format — the exact style these questions appear in interviews.

## What Interviewers Test
- Do you have a systematic diagnostic process, not just guesses?
- Can you correctly prioritize root causes by probability before investigating?
- Do you know the difference between training problems and data problems?
- Can you design experiments that isolate a single variable?
- Can you reason about serving issues that don't appear in training?

## The ML Debugging Framework

```
Step 1: Reproduce the problem
  → Quantify exactly: what's broken, by how much, when did it start?

Step 2: Scope the failure
  → Is it training, evaluation, or serving?
  → Did it fail from the start, or regress?

Step 3: Rank hypotheses by probability
  → Data > Features > Model > Code bug > Infrastructure
  → Start with the most common causes, not the most interesting

Step 4: Design minimal experiments
  → Change one thing at a time
  → Plot distributions, not just aggregate metrics

Step 5: Confirm the fix
  → Verify the metric recovers
  → Check that no other metric degrades (guardrails)
```

> 💡 **Interview Tip:** Interviewers want to see the framework, not the answer. Say "the first thing I'd check is..." not "I think it's a bug in...". Diagnostic reasoning beats confident guessing.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

np.random.seed(42)

# Helper for quick diagnostics
def check_feature_stats(X_train, X_serve, feature_names=None):
    """Check for distribution shifts between training and serving."""
    n = X_train.shape[1]
    if feature_names is None:
        feature_names = [f'feat_{i}' for i in range(n)]
    print(f"{'Feature':<12} {'Train mean':>11} {'Serve mean':>11} {'Train std':>10} {'Serve std':>10} {'PSI':>6}")
    print("-" * 65)
    for i, name in enumerate(feature_names):
        t_mean, t_std = X_train[:, i].mean(), X_train[:, i].std()
        s_mean, s_std = X_serve[:, i].mean(), X_serve[:, i].std()
        # Simple PSI approximation
        bins = np.linspace(min(X_train[:,i].min(), X_serve[:,i].min()),
                           max(X_train[:,i].max(), X_serve[:,i].max()) + 1e-6, 11)
        t_pct = np.clip(np.histogram(X_train[:,i], bins)[0] / len(X_train[:,i]), 1e-6, None)
        s_pct = np.clip(np.histogram(X_serve[:,i], bins)[0] / len(X_serve[:,i]), 1e-6, None)
        psi = float(np.sum((s_pct - t_pct) * np.log(s_pct / t_pct)))
        flag = "⚠️" if psi > 0.2 else ""
        print(f"{name:<12} {t_mean:>11.3f} {s_mean:>11.3f} {t_std:>10.3f} {s_std:>10.3f} {psi:>6.3f} {flag}")

# Simulate training vs serving data with a drifted feature
n_train, n_serve = 5000, 1000
X_train = np.random.randn(n_train, 4)
X_serve = X_train[:n_serve].copy()
X_serve[:, 2] = X_serve[:, 2] + 3.0   # feature 2 has drifted massively

print("Feature Distribution Check (training vs. serving):")
check_feature_stats(X_train, X_serve, feature_names=['user_age', 'session_len', 'item_price', 'ctr_7d'])


---
## Scenario 1: Model AUC Drops 8 Points After Retraining

**Interviewer:** You retrained your model on the last 30 days of data and AUC dropped from 0.87 to 0.79 on the holdout. What do you do?

**You:** First, I want to confirm this is a real regression and not an evaluation artifact. I'd check:

1. **Same holdout?** Is the holdout set the same split as before, or did it shift? If the holdout now covers a different time period or population, the comparison is invalid.
2. **Label distribution:** Is the positive rate in the new training data consistent with before? A drop in positive rate changes the task difficulty.
3. **Data volume:** Did the new training data shrink significantly? Less data → more variance in performance.

If evaluation setup is correct, then:
4. **Feature distributions:** Plot train vs. holdout feature histograms. If training distribution shifted, the holdout may be out-of-distribution.
5. **Learning curves:** Train on 10%, 30%, 100% of new data. Does AUC recover with more data? If yes, data quality issue in recent batch. If no, something structural changed.
6. **Sanity baseline:** What does the previous model get on the new holdout? If the old model also scores 0.79, the holdout distribution changed — not the model.

**Interviewer:** The old model also scores ~0.79 on the new holdout. What does that tell you?

**You:** The problem is the evaluation setup or the holdout data, not the model. The label or feature distribution in the new holdout period must have changed, making it harder. I'd look at class balance and feature distributions in the new holdout period vs. the old one.


## Scenario 2: Model Works Offline, Fails in Production

**Interviewer:** Your model gets AUC 0.91 in offline eval but the online A/B test shows CTR is identical to the control. No improvement at all. What's going on?

**You:** This is classic training-serving skew. The model looks great offline because offline training and evaluation share the same data pipeline. In production, something differs. My checklist:

1. **Feature computation code:** Is the feature computation in Python training identical to the C++/Java serving code? Even a small difference (different default values, different null handling, different bucketing) can cause skew.
2. **Feature freshness:** In training, did I use features computed at the "right" timestamp? In serving, are those features being looked up from a store with the right time granularity?
3. **Score distribution:** Log a sample of model scores from production. Are they in the same range as offline scores? If scores are compressed near 0.5, features are wrong.
4. **Label leakage:** In training, is there any feature that encodes the target? If so, AUC is inflated offline and the model has no real signal in production.
5. **Position bias:** If this is a ranking system, am I evaluating on position-biased clicks? High offline AUC on biased clicks means the model learned to score top-positioned items higher, not actually better items.

**Interviewer:** You log scores from production and they're all between 0.49 and 0.51. What do you conclude?

**You:** The model is getting garbage features. If all scores cluster near 0.5, the model has no useful input — it's outputting near-prior predictions. This is almost certainly a feature serving bug: null features, wrong feature names, or the feature store returning empty values. I'd add feature logging at serving time and compare the distributions to training.


## Scenario 3: Loss Explodes During Training

**Interviewer:** Your transformer model's loss NaN's out on step 400 of training. How do you debug this?

**You:** NaN loss during training has a handful of common causes, and I'd check them in order:

1. **Learning rate too high:** The most common cause. Try 10× smaller LR. NaN around step 300–500 is a sign the optimizer is taking huge steps after warmup ends.
2. **Exploding gradients:** Clip gradient norm before the step (e.g., `max_grad_norm=1.0`). Log the gradient norm each step — if it's growing exponentially, that's the issue.
3. **NaN in data:** A single NaN in a training batch propagates through all operations. Check: `assert not torch.isnan(batch).any()` before the forward pass.
4. **Numerical overflow:** Softmax of large logits overflows. Use numerically stable implementations (`F.cross_entropy` not `softmax + log + nll`).
5. **Log of zero:** Log-loss on zero-probability predictions. Add epsilon: `log(p + 1e-8)`.
6. **Mixed precision issues:** With fp16, small values underflow to 0 and cause log(0). Check loss scaling is configured correctly.

**Interviewer:** You add gradient clipping and the NaN moves to step 200. What do you do?

**You:** Gradient clipping stopped one explosion but the underlying instability is still there. I'd now: (a) reduce LR by 5× more; (b) add layer-wise gradient norm logging to find which layer is exploding first; (c) inspect the batch at step ~190 — is there a particularly difficult sample? If all samples at that batch have high loss, it could be a noisy label or an out-of-distribution example that's destabilizing training.


## Scenario 4: A/B Test Shows No Improvement

**Interviewer:** Your experiment shows treatment CTR = 4.21% and control CTR = 4.19%. Not significant. What do you conclude and what do you do next?

**You:** Before concluding the model doesn't work, I'd check the experiment's statistical validity:

1. **Is the test adequately powered?** If we're looking for a 1% relative lift (0.04 CTR points), we need enough users to detect it. Calculate: `n = (z_alpha + z_beta)^2 * p*(1-p) / (effect_size^2)`. If traffic is low, we may need more days.
2. **Did the treatment actually serve?** Check that 100% of treatment users received the new model, not a partial rollout or fallback.
3. **Novelty effect contamination?** If the test ran for only 3 days, early results may reflect novelty. Run for at least 7 days (2 full weekly cycles).
4. **Are we measuring the right metric?** CTR may not be the north-star. Check watch-time, session depth, long-term retention. A cold-start fix may show no CTR improvement but large retention improvement.
5. **Heterogeneous treatment effect:** Aggregate CTR hides subgroup differences. Segment by user age, device type, engagement level. The improvement may be significant for new users even if flat overall.

**Interviewer:** Segment analysis shows the new model improves CTR by 15% for new users (significant) but -1% for established users (significant). What do you do?

**You:** This is a classic precision vs. recall tradeoff at the model architecture level. The model helps new users but hurts established ones — probably because changes to the cold-start path interfere with the established user path. I'd explore: (a) separate models for new vs. established users; (b) a feature flag to route based on interaction count; (c) investigate why established users regress — is it a feature conflict, or does the new architecture change ranking in a way that hurts the established user distribution?


In [ ]:
# Statistical power calculator for A/B tests
from scipy import stats

def ab_test_power(control_rate, min_detectable_effect_relative,
                  alpha=0.05, power=0.8, two_sided=True):
    """Calculate required sample size per arm for a binary metric A/B test."""
    p1 = control_rate
    p2 = control_rate * (1 + min_detectable_effect_relative)
    
    z_alpha = stats.norm.ppf(1 - alpha / (2 if two_sided else 1))
    z_beta  = stats.norm.ppf(power)
    
    p_bar = (p1 + p2) / 2
    n = ((z_alpha * (2 * p_bar * (1-p_bar))**0.5 + z_beta * (p1*(1-p1) + p2*(1-p2))**0.5)
         / (p2 - p1))**2
    return int(np.ceil(n))

print("A/B Test Sample Size Requirements")
print(f"{'Metric':>12} {'Base Rate':>10} {'MDE':>6} {'N per arm':>10} {'Days @ 10K QPS':>16}")
print("-" * 60)

scenarios = [
    ('CTR',       0.042, 0.01),   # 1% relative lift
    ('CTR',       0.042, 0.05),   # 5% relative lift
    ('Conv. Rate',0.008, 0.05),   # 5% lift on rare event
    ('CTR',       0.042, 0.10),   # 10% relative lift
]

for metric, base, mde in scenarios:
    n = ab_test_power(base, mde)
    qps = 10000
    days = (2 * n) / (qps * 86400)
    print(f"{metric:>12} {base:>10.3f} {mde:>6.0%} {n:>10,} {days:>16.1f}d")


## Scenario 5: Model Score Distribution Has Shifted

**Interviewer:** You notice in monitoring that the model's mean output score has gone from 0.12 to 0.07 over the last week. No alerts fired. What do you do?

**You:** A 40% drop in mean score is significant. My first question: is this a model issue, a data issue, or expected behavior?

1. **Check the serving logs:** Did the input feature distribution change? Pull the last 1000 examples and compare feature histograms to the training distribution. If a key feature changed (e.g., time-of-day distribution shifted because traffic pattern changed), the scores would naturally shift.
2. **Check for upstream data pipeline changes:** Did any upstream data source change in the last week? Schema changes, ETL bugs, or new data source onboarding can change feature values silently.
3. **Check the label rate (if available):** If the true event rate decreased, the model lowering its scores might be correct. Query the actual outcome rate from the same period.
4. **Examine top examples:** Inspect the top 20 highest-scored examples. Do they look like genuine positives? Inspect the bottom 20. Are they clearly negative?
5. **Check for model rollback:** Did a model version change happen 7 days ago? A different model version might have a different score scale.

**Interviewer:** You find that 3 days ago, a feature called `user_activity_score` started being served as 0 for all users (a bug in the feature pipeline). What's your next step?

**You:** This is a training-serving skew bug. The feature was non-zero during training and is zero now. The model is getting incorrect features. Immediate action: fix the pipeline bug and restore the feature. If the fix takes >1 hour, consider reverting to a fallback model that doesn't use this feature, or applying a score correction. Then, add a data quality check to your monitoring: `user_activity_score` must have <10% zero rate; alert if it exceeds that.


## Scenario 6: Recommendation Model Degenerates Over Time

**Interviewer:** Three months after launch, you notice that your recommendation model is suggesting the same 50 videos to everyone, and engagement is falling. What happened and how do you fix it?

**You:** This is a classic feedback loop / popularity bias spiral. What likely happened:

1. The model learned that a set of popular videos had high engagement.
2. It recommended those videos more frequently.
3. Users saw and interacted with those videos more (because they were shown more, not because they're better).
4. This engagement was fed back into training data.
5. The model learned even more strongly that these 50 videos are good.
6. Repeat until the model only recommends a small cluster of popular content.

**How to detect:** Gini coefficient of recommended video distribution. If it's increasing toward 1.0, the recommendations are concentrating. Also, track unique videos recommended per day — should be a large fraction of the catalog, not 50.

**Short-term fixes:**
- Reduce the weight on recent engagement data (which is biased toward over-recommended items)
- Add a novelty/diversity reward to the objective (penalize repeated items)
- Enforce a minimum diversity constraint at re-ranking time (no more than X% from the same creator/category in top 20)

**Long-term fix:**
- Correct position/exposure bias in training data using inverse propensity weighting (IPW)
- Reserve 5–10% of recommendation slots for exploration (Thompson Sampling or ε-greedy)
- Log counterfactual data: for each unexplored item, log a "would have been recommended" score so you can correct the data distribution


## Key Takeaways
- Debugging framework: reproduce → scope → rank hypotheses by probability → isolate → confirm
- Data problems are 10× more common than model bugs — always check data first
- Training-serving skew: log feature distributions at serving time; compare to training
- NaN loss: learning rate first, then gradient clipping, then data NaN, then numerical precision
- A/B test: check power, treatment delivery, novelty effect, and segment before calling no-effect
- Feedback loops: popularity bias + high-engagement label → concentration; fix with IPW + exploration